# ML-007 · Подготовка данных для прогноза TCE

Цель — самостоятельно собрать входы для модели из журналов скважин, координат и TIFF. Открывай [справочник](СПРАВОЧНИК.md) по мере надобности. Заполняй ячейки с `TODO` и запускай проверку сразу после них. Проверка пропускается, пока нужные переменные или функции ещё не созданы.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import tifffile
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

here = Path.cwd().resolve()
repo = next(
    (folder for folder in (here, *here.parents) if (folder / "ML-006" / "practice_02").is_dir()),
    None,
)
if repo is None:
    raise FileNotFoundError("Открой ноутбук из ML-007 или из корня репозитория.")
DATA = repo / "ML-006" / "practice_02"

def check(required, verify):
    missing = [name for name in required.split() if name not in globals()]
    if missing:
        print("Сначала заполни TODO. Ожидаю:", ", ".join(missing))
        return
    verify()
    print("Проверка пройдена")

## 1. Аудит таблиц

Прочитай `wells.csv`, `concentration.csv`, `head.csv`, `cores.csv`, `pumping.csv`, `sources.csv` и `queries_test.csv` в `wells`, `raw`, `head`, `cores`, `pumping`, `sources`, `queries` соответственно. Сохрани ID скважины как строку при чтении. Посмотри на размеры, типы, число пропусков, число повторов и частоты `quality`. Найди два примера лабораторных значений, которые нельзя прямо привести к `float`.

**Подсказка.** `pd.read_csv`, `.info()`, `.isna().sum()`, `.duplicated()`, `.value_counts(dropna=False)`.

In [ ]:
# TODO: прочитай таблицы и выведи результаты аудита

In [ ]:
def verify_audit():
    assert wells.shape[0] == 262 and wells["well_id"].is_unique
    assert raw.shape[0] == 3731 and raw.duplicated().any()
    assert raw["quality"].eq("rejected").any()
    assert {"well_id", "t", "head"}.issubset(head.columns)
    assert {"z_top", "z_bottom", "lithology"}.issubset(cores.columns)
    assert set(pumping["units"]) == {"l/s", "m3/day"} and len(sources) == 3
    assert len(queries) == 960 and queries["query_id"].is_unique

check("wells raw head cores pumping sources queries", verify_audit)

## 2. Маски и единицы

Разминка. В `toy` исправь положительные глубины, используя `.loc`. Затем выведи строки, которые **не** лежат в диапазоне от −76 до 0. Сохрани результат во второй таблице `outside`.

В копии `pumping` создай `pump_clean` с новой колонкой `rate_m3_day`. Переведи л/с в м³/сут множителем 86,4. Для м³/сут множитель равен 1.

**Подсказка.** `~mask` инвертирует логические значения. `Series.map` подставляет число из словаря.

In [ ]:
toy = pd.DataFrame({"id": ["A", "B", "C"], "z": [12.0, -33.0, 90.0]})
# TODO: исправь toy, создай outside и pump_clean

In [ ]:
def verify_units():
    assert toy["z"].tolist() == [-12, -33, -90]
    assert outside["id"].tolist() == ["C"]
    in_ls = pumping["units"].eq("l/s")
    assert len(pump_clean) == len(pumping)
    assert np.allclose(pump_clean.loc[in_ls, "rate_m3_day"], pumping.loc[in_ls, "pumping_rate"] * 86.4)
    assert np.allclose(pump_clean.loc[~in_ls, "rate_m3_day"], pumping.loc[~in_ls, "pumping_rate"])

check("outside pump_clean", verify_units)

## 3. Даты и лабораторные замеры

Напиши `parse_dates(series)`. Она должна разбирать ISO и даты с точками. Нераспознанную дату обозначь `NaT`. Напиши `clean_samples(table)`, которая применяет следующие правила:

1. Оставь строки `ok` и `replicate`, убери полные дубли.
2. Приведи `t` к дате. Распознай `ND` и значения, которые начинаются с `<`. Для них возьми половину `lod_ug_l` и выстави `censored=True`.
3. Остальные значения преобразуй в числа. Удали строки без ID, с некорректной датой и с отсутствующим, бесконечным или отрицательным результатом.
4. Собери таблицу `samples` с колонками `well_id, t, concentration_ug_l, censored`. Одна строка на скважину и дату. При нескольких валидных пробах возьми медиану, а `censored` поставь в `True`, если хотя бы одна проба цензурирована.

**Подсказка.** `pd.to_datetime(format=..., errors="coerce")` дважды, `fillna`, `str.startswith`, `mask`, `groupby().agg()`. `NaT` — пропуск в колонке дат.

In [ ]:
# TODO: напиши parse_dates и clean_samples, затем вызови clean_samples(raw)

In [ ]:
def verify_samples():
    sample_dates = parse_dates(pd.Series(["2025-02-03", "03.02.2025", "bad"]))
    assert sample_dates.iloc[0] == sample_dates.iloc[1] == pd.Timestamp("2025-02-03")
    assert pd.isna(sample_dates.iloc[2])
    fixture = pd.DataFrame({
        "well_id": ["A", "A", "A", "A", "B", "C", "D", "E", None],
        "t": ["2025-02-03", "2025-02-03", "03.02.2025", "2025-02-03",
              "2025-02-03", "bad", "2025-02-03", "2025-02-03", "2025-02-03"],
        "concentration": ["ND", "ND", "1.9", "99", "<0.05", "1", "-1", "inf", "1"],
        "lod_ug_l": [0.2, 0.2, 0.2, 0.2, 0.05, 0.05, 0.05, 0.05, 0.05],
        "quality": ["ok", "ok", "replicate", "rejected", "ok", "ok", "ok", "ok", "ok"],
    })
    cleaned = clean_samples(fixture).set_index("well_id")
    assert set(cleaned.index) == {"A", "B"} and cleaned.index.is_unique
    assert np.isclose(cleaned.at["A", "concentration_ug_l"], 1.0)
    assert np.isclose(cleaned.at["B", "concentration_ug_l"], 0.025)
    assert cleaned["censored"].all()
    assert not samples.duplicated(["well_id", "t"]).any()
    assert samples["t"].notna().all() and len(samples) == 3616
    assert np.isfinite(samples["concentration_ug_l"]).all()

check("parse_dates clean_samples samples", verify_samples)

## 4. Присоединение координат и литологии

Присоедини координаты из `wells` по `well_id` к каждой строке `samples`. Сохрани таблицу в `located`, сохранив ключ пробы `(well_id, t)`. Проверь кратность объединения через `validate`.

Присоедини слои из `cores` по `well_id`. Оставь для каждой пробы подходящий слой по условию `z_bottom < z <= z_top` и сохрани в `layered`. Проверь, что ни одна проба из этого набора не пропала. На границе `z = -20` проба относится к слою, который начинается на −20 м.

**Подсказка.** После первого `merge` число строк сохраняется. После второго временно станет больше: у каждой скважины три слоя.

In [ ]:
# TODO: собери located и layered

In [ ]:
def verify_location():
    key = ["well_id", "t"]
    assert len(located) == len(layered) == len(samples)
    assert not located.duplicated(key).any() and not layered.duplicated(key).any()
    assert located[["x", "y", "z"]].notna().all().all()
    assert layered["lithology"].notna().all()
    assert ((layered["z_bottom"] < layered["z"]) & (layered["z"] <= layered["z_top"])).all()

check("located layered samples", verify_location)

## 5. Предыдущий замер напора

Собери `with_head` из `layered` и `head`. В `head` разбери даты и переименуй `t` в `head_t`. Для каждой пробы найди **последний доступный** напор той же скважины на момент пробы. Замер того же дня подходит. Разрешённый разрыв — до 180 суток включительно. Если подходящего замера нет, оставь пропуск.

**Подсказка.** `merge_asof` с `by="well_id"`, `direction="backward"`, `tolerance=pd.Timedelta(days=180)`. Сортируй обе таблицы по колонкам времени. В следующих заданиях таблица `with_head` остаётся для аудита. Запросы содержат только координаты и дату, поэтому для них нужен свой способ оценивать напор.

In [ ]:
# TODO: собери with_head

In [ ]:
def verify_head():
    assert len(with_head) == len(layered)
    assert not with_head.duplicated(["well_id", "t"]).any()
    assert with_head["head_t"].notna().any() and with_head["head_t"].isna().any()
    gaps = (with_head["t"] - with_head["head_t"]).dropna()
    assert gaps.between(pd.Timedelta(0), pd.Timedelta(days=180)).all()
    audit = with_head[["well_id", "t", "head_t"]].sample(25, random_state=42)
    history = head.assign(available_t=parse_dates(head["t"]))
    for row in audit.itertuples(index=False):
        match = history.loc[
            history["well_id"].eq(row.well_id)
            & history["available_t"].between(row.t - pd.Timedelta(days=180), row.t),
            "available_t",
        ].max()
        assert (pd.isna(match) and pd.isna(row.head_t)) or match == row.head_t

check("with_head layered head parse_dates", verify_head)

## 6. Растр, `np.mgrid` и общие признаки

**Разминка.** Для `demo` построй `row_grid` и `col_grid` через `np.mgrid`. Выведи их и проверь, что `demo[row_grid, col_grid]` возвращает исходную матрицу.

**Задача.** Прочитай `permeability.tif` в `raster`. Напиши `sample_raster(x, y)`, которая принимает одномерные массивы координат и возвращает проводимость в м/сут. У ячейки шаг 50 м и начало сетки `(0, 0)`. У `x` индекс столбца, у `y` индекс строки. При координатах за границами, неконечных координатах или `-9999` верни `np.nan`.

Напиши `make_features(points)` для таблицы с `x, y, z, t`. Сохрани порядок строк и исходный индекс. Выход должен содержать колонки в порядке `NUM + CAT`:

- `x, y, z` как есть и `days` от 1 января 2020 года
- `k_m_day` из растра
- `source_dist_m` — минимальное расстояние в плоскости до точки из `sources.csv`
- `depth_band` — `deep` при `z <= -46`, `middle` при `-46 < z <= -20`, иначе `shallow`

**Подсказка.** `floor` переводит метры в индекс. Проверяй границы до обращения к массиву. Для расстояний можно посчитать `np.hypot` до каждого источника и взять минимум по источникам. Для глубины подойдёт `pd.cut`.

In [ ]:
demo = np.array([[10, 11, 12], [20, 21, 22]])
domain = json.loads((DATA / "domain.json").read_text(encoding="utf-8"))
NUM = ["x", "y", "z", "days", "k_m_day", "source_dist_m"]
CAT = ["depth_band"]
# TODO: создай сетки, прочитай raster, напиши sample_raster и make_features

In [ ]:
def verify_raster():
    assert np.array_equal(row_grid, [[0, 0, 0], [1, 1, 1]])
    assert np.array_equal(col_grid, [[0, 1, 2], [0, 1, 2]])
    assert np.array_equal(demo[row_grid, col_grid], demo)
    assert raster.shape == (domain["ny"], domain["nx"]) == (60, 80)
    values = sample_raster(np.array([525., 25., -1., 4000., np.nan]),
                           np.array([275., 25., 275., 275., 275.]))
    assert np.isclose(values[0], raster[5, 10]) and np.isnan(values[1:]).all()
    probe = pd.DataFrame({"x": [525., 0., 0.], "y": [275., 0., 0.], "z": [-20., -46., -3.],
                          "t": pd.to_datetime(["2020-01-02", "2020-01-01", "2020-01-01"])},
                         index=[8, 3, 11])
    features = make_features(probe)
    assert features.index.equals(probe.index) and list(features.columns) == NUM + CAT
    assert features["days"].tolist() == [1, 0, 0]
    assert features["depth_band"].tolist() == ["middle", "deep", "shallow"]
    assert np.isclose(features.at[8, "k_m_day"], raster[5, 10])
    assert np.isclose(features.at[8, "source_dist_m"], np.hypot(sources.x - 525, sources.y - 275).min())

check("row_grid col_grid raster sample_raster make_features sources", verify_raster)

## 7. Временное разбиение и единая схема

Раздели `located` на `train` с датами до 1 января 2025 и `val` с датами начиная с 1 января 2025. Применяй `make_features` к ним и к копии `queries` с разобранным временем. Получи `X_train`, `X_val`, `X_query`. Вынеси концентрацию в `y_train`, `y_val` с теми же индексами. Для запросов сохрани отдельную серию `query_ids`. Все запросы оставь в исходном порядке.

**Подсказка.** Проверь даты, колонки и индексы до преобразования. При желании отдельно сравни с разбиением по скважинам через `GroupShuffleSplit`.

In [ ]:
cutoff = pd.Timestamp("2025-01-01")
# TODO: создай train, val, X_train, X_val, X_query, y_train, y_val, query_ids

In [ ]:
def verify_split():
    assert len(train) + len(val) == len(located)
    assert train["t"].max() < cutoff <= val["t"].min()
    assert list(X_train.columns) == list(X_val.columns) == list(X_query.columns) == NUM + CAT
    assert X_train.index.equals(y_train.index) and X_val.index.equals(y_val.index)
    assert y_train.equals(train["concentration_ug_l"]) and y_val.equals(val["concentration_ug_l"])
    assert len(X_query) == len(queries) and X_query.index.equals(query_ids.index)
    assert query_ids.tolist() == queries["query_id"].tolist()

check("train val X_train X_val X_query y_train y_val query_ids", verify_split)

## 8. Масштабирование и кодирование без утечки

Создай `prep` через `ColumnTransformer`. Назови числовой блок `num` и собери в нём `SimpleImputer(strategy="median")` и `StandardScaler`. Категориальный блок назови `cat` и используй `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`. Обучи `prep` только на `X_train`. Получи три числовых массива `X_train_ready`, `X_val_ready`, `X_query_ready`.

Сравни среднее, минимум и максимум для `x` до и после стандартизации. Попробуй передать одну искусственную строку с незнакомой категорией и пропуском проводимости.

**Дополнительно.** Обучи `KNeighborsRegressor(n_neighbors=10)` дважды. В первом запуске используй только числовые признаки и заполни пропуски без масштабирования. Во втором используй те же числовые признаки после стандартизации. Сравни `root_mean_squared_error` на одном отложенном годе и объясни, почему изменились ближайшие соседи.

In [ ]:
# TODO: создай prep и три преобразованные матрицы

In [ ]:
def verify_preprocessing():
    from sklearn.utils.validation import check_is_fitted

    check_is_fitted(prep)
    for ready, original in [(X_train_ready, X_train), (X_val_ready, X_val), (X_query_ready, X_query)]:
        assert ready.shape == (len(original), X_train_ready.shape[1])
        assert np.isfinite(ready).all()
        assert np.allclose(ready, prep.transform(original))
    numeric = prep.named_transformers_["num"]
    imputer, scaler = numeric.steps[0][1], numeric.steps[1][1]
    assert np.allclose(imputer.statistics_, X_train[NUM].median().to_numpy())
    assert np.allclose(scaler.mean_, imputer.transform(X_train[NUM]).mean(axis=0))
    assert np.allclose(X_train_ready[:, :len(NUM)].mean(axis=0), 0, atol=1e-9)
    probe = X_query.iloc[:1].copy()
    probe["depth_band"] = "unknown"
    probe["k_m_day"] = np.nan
    transformed = prep.transform(probe)
    assert np.isfinite(transformed).all()
    assert np.allclose(transformed[:, len(NUM):], 0)

check("prep X_train_ready X_val_ready X_query_ready", verify_preprocessing)

## Повтори по памяти

В пустой ячейке выполни три операции на маленькой таблице: отфильтруй брак через `~`, присоедини координаты через `merge` и стандартизируй один столбец. Когда сможешь объяснить, почему `fit` делается на обучении и почему TIFF индексируется `[row, col]`, возвращайся к полному варианту.